# 🔥 Forest Fire Risk Prediction

Predict unplanned forest fires across Australia using tenure type, forest category, and prior fire history.

**Data:** ABARES Forest Fire 2016–2021 (agriculture.gov.au/abares/forestsaustralia)

**Model:** Random Forest Classifier

**Model:** deepseek-v4-pro

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, roc_auc_score,
                             confusion_matrix, RocCurveDisplay,
                             accuracy_score, precision_score,
                             recall_score, f1_score)
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
print('✅ Libraries loaded')

## Load & Parse Data

In [ ]:
# Load ABARES forest fire data
csv = Path('../../daily-datasets/climate/bushfire-history/raw/Fire_For16-21_Attributes.csv')
df = pd.read_csv(csv)
print(f'Loaded {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head(3)

In [ ]:
# Decode fire patterns from ALL_FIRE column
def parse_fire(s):
    s = str(s).strip().ljust(5)
    return [1 if i < len(s) and s[i] == 'U' else 0 for i in range(5)]

fire_cols = df['ALL_FIRE'].apply(parse_fire)
fire_df = pd.DataFrame(fire_cols.tolist(), columns=[f'unplanned_{i}' for i in range(1, 6)])
df2 = pd.concat([df, fire_df], axis=1)
df2['any_unplanned_prior'] = df2[[f'unplanned_{i}' for i in range(1, 5)]].sum(axis=1).clip(0, 1)
df2['total_burns'] = df2['FOR_BURNS'].clip(0, 5)

print(f'Decoded {len(fire_df)} fire patterns')
print(f'Unplanned fire rate (2020-21): {df2["unplanned_5"].mean()*100:.1f}%')
df2[['STATE', 'FOR_TEN', 'FOR_CATEGO', 'FOR_BURNS', 'unplanned_5']].head()

## Exploratory Data Analysis

In [ ]:
# Filter to forest regions only
forest = df2[df2['FOREST'] == 1].copy()
burns = forest[forest['FOR_BURNS'] >= 0].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Forest Fire Data Overview', fontsize=14, fontweight='bold')

# 1) Pixel count by state
state_counts = forest['STATE'].value_counts()
axes[0].barh(state_counts.index, state_counts.values,
             color=sns.color_palette('Reds_r', len(state_counts)))
axes[0].set_xlabel('Pixels (thousands)')
axes[0].set_title('Forest Regions by State')

# 2) Burn count distribution
sns.histplot(burns['FOR_BURNS'], bins=6, discrete=True, ax=axes[1], color='#d9534f')
axes[1].set_xlabel('Number of Burns (2016-2021)')
axes[1].set_ylabel('Pixel Count')
axes[1].set_title('Fire Frequency Distribution')

# 3) Tenure type breakdown
tenure_counts = forest['FOR_TEN'].value_counts()
axes[2].pie(tenure_counts.values, labels=tenure_counts.index, autopct='%1.1f%%',
            colors=sns.color_palette('YlOrRd', len(tenure_counts)), startangle=90)
axes[2].set_title('Land Tenure Breakdown')

plt.tight_layout()
plt.show()

In [ ]:
# Fire rate by state and tenure
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

state_avg = burns.groupby('STATE')['FOR_BURNS'].mean().sort_values()
colors = ['#d9534f' if v > 1.5 else '#f0ad4e' if v > 0.5 else '#5cb85c' for v in state_avg.values]
axes[0].barh(state_avg.index, state_avg.values, color=colors)
axes[0].set_xlabel('Mean Burns (2016-2021)')
axes[0].set_title('Average Fire Frequency by State')

ten_avg = burns.groupby('FOR_TEN')['FOR_BURNS'].mean().sort_values()
colors2 = ['#d9534f' if v > 2 else '#f0ad4e' if v > 1 else '#5cb85c' for v in ten_avg.values]
axes[1].barh(ten_avg.index, ten_avg.values, color=colors2)
axes[1].set_xlabel('Mean Burns (2016-2021)')
axes[1].set_title('Average Fire Frequency by Tenure Type')

plt.tight_layout()
plt.show()

In [ ]:
# State × Tenure heatmap of unplanned fire rate
heat = burns.groupby(['STATE', 'FOR_TEN'])['unplanned_5'].mean().unstack()

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(heat, annot=True, fmt='.2f', cmap='YlOrRd', linewidths=0.5,
            cbar_kws={'label': 'Unplanned Fire Rate (2020-21)'})
ax.set_title('Wildfire Risk: State × Tenure Type', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Burn count distributions
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(data=burns, x='FOR_TEN', y='FOR_BURNS', ax=axes[0],
            palette='YlOrRd', order=sorted(burns['FOR_TEN'].unique()))
axes[0].set_title('Burn Count Distribution by Tenure', fontweight='bold')

sns.boxplot(data=burns, x='STATE', y='FOR_BURNS', ax=axes[1],
            palette='RdYlGn_r', order=sorted(burns['STATE'].unique()))
axes[1].set_title('Burn Count Distribution by State', fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## Machine Learning: Random Forest Classifier

In [ ]:
# Feature engineering
ml = burns.copy()

features = pd.get_dummies(ml[['STATE', 'FOR_TEN', 'FOR_CATEGO']], drop_first=False).astype(int)
features['prior_burns'] = ml['FOR_BURNS']
features['any_unplanned_prior'] = ml['any_unplanned_prior']

# Target: unplanned fire in 2020-21
y = ml['unplanned_5'].values
X = np.nan_to_num(features.values, nan=0.0)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model = RandomForestClassifier(n_estimators=200, max_depth=12,
                               min_samples_leaf=50, class_weight='balanced',
                               random_state=42, n_jobs=-1)
model.fit(X_train_s, y_train)

y_pred = model.predict(X_test_s)
y_proba = model.predict_proba(X_test_s)[:, 1]
auc = roc_auc_score(y_test, y_proba)
cm = confusion_matrix(y_test, y_pred)

print(f'Training samples: {X_train.shape[0]:,}')
print(f'Test samples: {X_test.shape[0]:,}')
print(f'Features: {X_train.shape[1]}')
print(f'Class balance: {y.mean()*100:.1f}% positive')
print(f'ROC-AUC: {auc:.4f}')

# Feature importance
imp = pd.DataFrame({'feature': features.columns, 'importance': model.feature_importances_})
imp = imp.sort_values('importance', ascending=False).head(15)

In [ ]:
# Confusion Matrix
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['No Fire', 'Fire'], yticklabels=['No Fire', 'Fire'])
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix — Random Forest', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance
fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#d9534f' if i < 3 else '#f0ad4e' if i < 6 else '#5cb85c' for i in range(len(imp))]
sns.barplot(data=imp, y='feature', x='importance', palette=colors)
ax.set_xlabel('Importance')
ax.set_title('Top 15 Feature Importances', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curve
fig, ax = plt.subplots(figsize=(7, 6))
RocCurveDisplay.from_predictions(y_test, y_proba, ax=ax, color='#d9534f', linewidth=2)
ax.plot([0,1], [0,1], 'k--', alpha=0.5)
ax.set_title(f'ROC Curve (AUC = {auc:.4f})', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 2020-21 Fire rate by State and Tenure
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

rate_state = ml.groupby('STATE')['unplanned_5'].mean().sort_values()
axes[0].bar(rate_state.index, rate_state.values, color=sns.color_palette('Reds', len(rate_state)))
axes[0].set_title('2020-21 Unplanned Fire Rate by State', fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

rate_ten = ml.groupby('FOR_TEN')['unplanned_5'].mean().sort_values()
axes[1].bar(rate_ten.index, rate_ten.values, color=sns.color_palette('Oranges', len(rate_ten)))
axes[1].set_title('2020-21 Unplanned Fire Rate by Tenure', fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Top 15 highest-risk region-tenure combinations
high_risk = ml[ml['unplanned_5'] == 1]
top = high_risk.groupby(['STATE', 'FOR_TEN']).size().reset_index(name='count')
top = top.sort_values('count', ascending=False).head(15)
top['label'] = top['STATE'] + ' — ' + top['FOR_TEN']

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=top, y='label', x='count', palette='Reds_r')
ax.set_xlabel('Number of High-Risk Regions')
ax.set_title('Top 15 Highest-Risk Region-Tenure Combinations', fontweight='bold')
plt.tight_layout()
plt.show()

## Model Performance Summary

In [ ]:
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print('=' * 50)
print('MODEL PERFORMANCE SUMMARY')
print('=' * 50)
print(f'ROC-AUC:      {auc:.4f}')
print(f'Accuracy:     {acc:.4f}')
print(f'Precision:    {prec:.4f}')
print(f'Recall:       {rec:.4f}')
print(f'F1-Score:     {f1:.4f}')
print(f'Features:     {model.n_features_in_}')
print(f'Trees:        {model.n_estimators}')
print(f'Fire rate:    {ml["unplanned_5"].mean()*100:.1f}%')
print(f'Train:        {X_train.shape[0]:,}')
print(f'Test:         {X_test.shape[0]:,}')